# Sinhala TTS — VITS Training on Kaggle

Train a [Coqui TTS](https://github.com/coqui-ai/TTS) **VITS** model for Sinhala speech synthesis using Kaggle's free T4 GPU.

**Setup:**
1. Enable GPU: Settings > Accelerator > GPU T4 x2
2. Add dataset: `keshan/multi-speaket-tts-dataset-sinhala`
3. Run all cells

**Session limits:** 12 hours max. Checkpoints are saved to `/kaggle/working/` for download and resume.

In [ ]:
# Cell 1: Install dependencies
!pip install -q coqui-tts==0.27.5 soundfile
!pip install -q --force-reinstall --no-deps numpy==2.2.6 numba==0.61.2 llvmlite==0.44.0 librosa==0.11.0 transformers==4.57.6

import torch
print(f"torch: {torch.__version__}, CUDA: {torch.cuda.is_available()}, GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")

In [ ]:
# Cell 2: Locate dataset and build LJSpeech layout
import json
import random
import shutil
from pathlib import Path

# Kaggle datasets are mounted at /kaggle/input/
DATASET_ROOT = Path("/kaggle/input/multi-speaket-tts-dataset-sinhala")
OUTPUT_DIR = Path("/kaggle/working/sinhala_ljspeech")
RUN_DIR = Path("/kaggle/working/runs/vits_sinhala")
SPEAKER_KEEP = "01"

# Find file-mapping.json
mapping_file = DATASET_ROOT / "file-mapping.json"
if not mapping_file.exists():
    mapping_file = next(DATASET_ROOT.rglob("file-mapping.json"), None)
assert mapping_file and mapping_file.exists(), "file-mapping.json not found!"
ds_root = mapping_file.parent
print(f"Dataset root: {ds_root}")

In [ ]:
# Cell 3: Build LJSpeech-style dataset
raw = json.loads(mapping_file.read_text(encoding="utf-8"))

wavs_dir = OUTPUT_DIR / "wavs"
wavs_dir.mkdir(parents=True, exist_ok=True)

PUNCTUATIONS = set(" .,!?:;\"'()-/")
rows = []
all_chars = set()
missing = 0

for key, rec in raw.items():
    newfn = rec.get("newfn")
    text = (rec.get("text") or "").strip()
    if not newfn or not text:
        continue
    # Filter by speaker
    stem = Path(newfn).stem
    parts = stem.split("_")
    spk = parts[1] if len(parts) >= 3 and parts[0] == "sin" else (parts[2] if len(parts) >= 4 and parts[0] == "pn" and parts[1] == "sin" else "unknown")
    if SPEAKER_KEEP and spk != SPEAKER_KEEP:
        continue
    # Resolve audio path
    fn = Path(newfn).name
    pn_fn = fn if fn.startswith("pn_") else f"pn_{fn}"
    src = None
    for candidate in [ds_root / fn, ds_root / pn_fn]:
        if candidate.exists():
            src = candidate
            break
    if src is None:
        for pattern in [fn, pn_fn]:
            found = next(ds_root.rglob(pattern), None)
            if found:
                src = found
                break
    if src is None:
        missing += 1
        continue
    target = wavs_dir / fn
    if not target.exists():
        shutil.copyfile(src, target)
    file_id = target.stem
    rows.append(f"{file_id}|{text}|{text}")
    all_chars.update(text)

all_chars -= PUNCTUATIONS
chars_str = "".join(sorted(all_chars))

random.seed(42)
random.shuffle(rows)
split = int(len(rows) * 0.95)
(OUTPUT_DIR / "metadata_train.txt").write_text("\n".join(rows[:split]), encoding="utf-8")
(OUTPUT_DIR / "metadata_val.txt").write_text("\n".join(rows[split:]), encoding="utf-8")

print(f"Total clips: {len(rows)}, Missing: {missing}")
print(f"Train: {split}, Val: {len(rows) - split}")
print(f"Characters ({len(chars_str)}): {chars_str[:60]}...")

In [ ]:
# Cell 4: Create VITS config
from TTS.tts.configs.shared_configs import BaseDatasetConfig, CharactersConfig
from TTS.tts.configs.vits_config import VitsConfig
from TTS.tts.models.vits import VitsAudioConfig

RUN_DIR.mkdir(parents=True, exist_ok=True)

dataset_cfg = BaseDatasetConfig(
    dataset_name="sinhala",
    path=str(OUTPUT_DIR),
    meta_file_train="metadata_train.txt",
    meta_file_val="metadata_val.txt",
    formatter="ljspeech",
)

characters = CharactersConfig(
    characters_class="TTS.tts.utils.text.characters.Graphemes",
    characters=chars_str,
    punctuations=" .,!?:;\"'()-/",
    phonemes="",
    pad="<PAD>",
    eos="<EOS>",
    bos="<BOS>",
    blank="<BLNK>",
)

audio = VitsAudioConfig(sample_rate=22050)

# Kaggle T4: batch_size=8, eval_batch_size=4
config = VitsConfig(
    audio=audio,
    run_name="vits_sinhala",
    output_path=str(RUN_DIR),
    datasets=[dataset_cfg],
    use_phonemes=False,
    text_cleaner="basic_cleaners",
    characters=characters,
    batch_size=8,
    eval_batch_size=4,
    max_audio_len=500000,
    epochs=100,
    mixed_precision=True,
    num_loader_workers=2,
    num_eval_loader_workers=1,
    run_eval_steps=1000,
    save_step=1000,
    print_step=250,
)

cfg_path = RUN_DIR / "config.json"
config.save_json(str(cfg_path))
print(f"Config saved to: {cfg_path}")

In [ ]:
# Cell 5: Train!
# To resume from a checkpoint, uncomment the --restore_path line below.
import sys

sys.argv = ["train_tts", "--config_path", str(cfg_path)]
# sys.argv += ["--restore_path", str(RUN_DIR / "<run_folder>" / "best_model_XXXX.pth")]

from TTS.bin.train_tts import main
main()

In [ ]:
# Cell 6: Test inference
from TTS.utils.synthesizer import Synthesizer
import numpy as np
import soundfile as sf
from IPython.display import Audio

# Find the best model
import glob
best_models = sorted(glob.glob(str(RUN_DIR / "*" / "best_model_*.pth")))
if best_models:
    ckpt = best_models[-1]
    cfg = str(Path(ckpt).parent / "config.json")
    print(f"Using: {ckpt}")
    
    synth = Synthesizer(tts_checkpoint=ckpt, tts_config_path=cfg, use_cuda=True)
    wav = synth.tts("ආයුබෝවන්")
    sf.write("/kaggle/working/output.wav", np.array(wav), synth.tts_config.audio.sample_rate)
    Audio(np.array(wav), rate=synth.tts_config.audio.sample_rate)
else:
    print("No best model found yet.")

## Saving & Resuming

Before your session expires:
1. Download checkpoints from `/kaggle/working/runs/`
2. In your next session, upload them back and set `--restore_path` in Cell 5

Or use Kaggle's **Save & Run All** to auto-save outputs.